In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
from qdrant_client import QdrantClient


c:\Users\Loq\Documents\CRAP\end to end aibootcamp\code\handsON\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3046.92it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
import logging
# This disables INFO and DEBUG logging which usually triggers progress bars
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

In [35]:
import transformers
transformers.logging.set_verbosity_error()


In [2]:
import groq

### Embedding function

In [34]:
def get_embedding(text,model_name="all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    return model.encode(text,show_progress_bar=False).tolist()

### Retrival function

In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [6]:
def retrieve_data(query,qdrant_client, top_k=5):

    query_embedding=get_embedding(query)

    search_result=qdrant_client.query_points(
        collection_name="amazon-items-collection-00",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]
    for search_result in search_result.points:
        retrieved_context_ids.append(search_result.payload["parent_asin"])
        retrieved_context.append(search_result.payload["description"])
        retrieved_context_ratings.append(search_result.payload["average_rating"])
        similarity_scores.append(search_result.score)


    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

In [9]:
retrieved_context=retrieve_data("Which album is of hiphop genre?",qdrant_client, top_k=10)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8557.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
retrieved_context

{'retrieved_context_ids': ['B09MVH85ZP',
  'B0BJP8ZBRT',
  'B09WGLV528',
  'B09XVCWQGL',
  'B09PVHGSRZ',
  'B09RQZ7S38',
  'B09W9SVX44',
  'B09ZLQW9ZT',
  'B009A9BGBE',
  'B09Z6TY5F1'],
 'retrieved_context': ["Now That's What I Call Punk & New Wave / Various Limited Pink Double vinyl LP pressing. NOW Music proudly presents NOW That's What I Call Punk & New Wave - a collection encapsulating the spirit of the times, and of the movement that changed the course of mainstream pop music. 34 tracks across 2 LPs, present the era's most iconic artists, including The Clash, Blondie, Ramones, U2, Siouxsie And The Banshees, Adam & The Ants, The Jam and The Cure. The late 70s and early 80s saw an explosion of punk and new wave groups and artists moving from the underground to mainstream chart success. No.1s from The Jam with 'Going Underground', 'Brass In Pocket' by Pretenders, The Boomtown Rats' 'Rat Trap', Dexys Midnight Runners' 'Geno', and Ian Dury & The Blockheads' 'Hit Me With Your Rhythm Sti

### format retrieved context function

In [14]:
def process_context(context):
    formatted_context=""

    for id,chunk,rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context+=f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

In [15]:
preprocessed_context=process_context(retrieved_context)

In [16]:
print(preprocessed_context)

- ID: B09MVH85ZP, rating: 4.7, description: Now That's What I Call Punk & New Wave / Various Limited Pink Double vinyl LP pressing. NOW Music proudly presents NOW That's What I Call Punk & New Wave - a collection encapsulating the spirit of the times, and of the movement that changed the course of mainstream pop music. 34 tracks across 2 LPs, present the era's most iconic artists, including The Clash, Blondie, Ramones, U2, Siouxsie And The Banshees, Adam & The Ants, The Jam and The Cure. The late 70s and early 80s saw an explosion of punk and new wave groups and artists moving from the underground to mainstream chart success. No.1s from The Jam with 'Going Underground', 'Brass In Pocket' by Pretenders, The Boomtown Rats' 'Rat Trap', Dexys Midnight Runners' 'Geno', and Ian Dury & The Blockheads' 'Hit Me With Your Rhythm Stick' are all here, as well as top 5 hits from Squeeze, Tom Robinson Band and Patti Smith. The genre also gave the first crossover success for bands that became the big

### create a prompt for the LLM using the retrieved context and the user query

In [17]:
### create a prompt for the LLM using the retrieved context and the user query


def build_prompt(preprocessed_context, question):
    prompt=f"""
    You are a shopping assistant that can answer questions about the products in the stock.

    You will be given a question and a list of context.

    Instructions:
    -You need to answer the question based on the provided context only.
    -never use the word context and refer to it as the available products.

    Context:    
    {preprocessed_context}

    Question: {question}
""" 
    return prompt

In [23]:
prompt=build_prompt(preprocessed_context, "Which album is of hiphop genre? and has a rating higher than 4.5?")

In [24]:
print(prompt)


    You are a shopping assistant that can answer questions about the products in the stock.

    You will be given a question and a list of context.

    Instructions:
    -You need to answer the question based on the provided context only.
    -never use the word context and refer to it as the available products.

    Context:    
    - ID: B09MVH85ZP, rating: 4.7, description: Now That's What I Call Punk & New Wave / Various Limited Pink Double vinyl LP pressing. NOW Music proudly presents NOW That's What I Call Punk & New Wave - a collection encapsulating the spirit of the times, and of the movement that changed the course of mainstream pop music. 34 tracks across 2 LPs, present the era's most iconic artists, including The Clash, Blondie, Ramones, U2, Siouxsie And The Banshees, Adam & The Ants, The Jam and The Cure. The late 70s and early 80s saw an explosion of punk and new wave groups and artists moving from the underground to mainstream chart success. No.1s from The Jam with 'Go

### Generate Answer function

In [28]:
### Generate Answer function

from groq import Groq
client = Groq()
def generate_answer(prompt):


    completion = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
        {
            "role": "system",
            "content": prompt
        }
        ],
        reasoning_effort="default",
        reasoning_format="hidden"
    )
    return completion.choices[0].message.content

In [29]:
print(generate_answer(prompt))

The album **DoggyStyle** by Snoop Dogg (ID: B09WGLV528) belongs to the hip-hop genre and has a rating of **4.8**. It is described as a significant hip-hop album that helped introduce g-funk to a mainstream audience. The vinyl features a black/red with yellow/blue splatter design.


### combined rag pipeline

In [30]:
def rag_pipeline(query,top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context=retrieve_data(query,qdrant_client, top_k)
    preprocessed_context=process_context(retrieved_context)
    prompt=build_prompt(preprocessed_context, query)
    answer=generate_answer(prompt)
    return answer

In [36]:
print(rag_pipeline("What kind of albums can i get with a rating higher than 4.5 and which have high bass punch and edm genre ?"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7030.78it/s]


Based on the provided descriptions, the album **"Front by Front" by Front 242 (ID: B009A9BGBE)** fits your criteria. It is described as a foundational EBM (Electronic Body Music) album, a subgenre of industrial and electronic music, which aligns with the request for a "high bass punch" and electronic/EDM characteristics. It has a **4.7 rating**. 

No other albums in the given context explicitly mention EDM or bass-heavy electronic elements.


In [1]:
from groq import Groq
client = Groq()
def generate_answer():


    completion = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
        {
            "role": "system",
            "content": "write a gud essay in 100 words"
        }
        ],
        reasoning_effort="default",
        reasoning_format="hidden"
    )
    return completion

In [3]:
six=generate_answer()

In [4]:
six

ChatCompletion(id='chatcmpl-3a968401-bd62-414e-9df2-c1fa3c79fd6e', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Perseverance is the key to unlocking potential. History shows that progress thrives through resilience—Thomas Edison failed over 1,000 times before perfecting the light bulb. Each failure taught him what didn’t work, guiding his final breakthrough. Personal success mirrors this journey: setbacks are lessons, not dead ends. Challenges forge determination, teaching adaptability and grit. Whether in education, careers, or passions, persistence turns obstacles into opportunities. Success isn’t about avoiding failure but embracing the process of growth. As Edison said, “Our greatest glory is not in never falling, but in rising every time we fall.” By persisting through adversity, we not only achieve goals but also build unshakeable confidence. In a world of constant change, perseverance is the foundation of both personal triu

In [ ]:
ChatCompletion(id='chatcmpl-3a968401-bd62-414e-9df2-c1fa3c79fd6e', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Perseverance is the key to unlocking potential. History shows that progress thrives through resilience—Thomas Edison failed over 1,000 times before perfecting the light bulb. Each failure taught him what didn’t work, guiding his final breakthrough. Personal success mirrors this journey: setbacks are lessons, not dead ends. Challenges forge determination, teaching adaptability and grit. Whether in education, careers, or passions, persistence turns obstacles into opportunities. Success isn’t about avoiding failure but embracing the process of growth. As Edison said, “Our greatest glory is not in never falling, but in rising every time we fall.” By persisting through adversity, we not only achieve goals but also build unshakeable confidence. In a world of constant change, perseverance is the foundation of both personal triumph and collective progress. (99 words)', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1774784342, model='qwen/qwen3-32b', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_5cf921caa2', usage=CompletionUsage(completion_tokens=497, prompt_tokens=19, total_tokens=516, completion_time=1.284815074, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=324), prompt_time=0.000574971, prompt_tokens_details=None, queue_time=0.046935688, total_time=1.285390045), usage_breakdown=None, x_groq=XGroq(id='req_01kmwp7rxze4gsf3p37g1c9nks', debug=None, seed=315993274, usage=None))

In [7]:
six.usage.completion_tokens

497

In [ ]:
CompletionUsage(completion_tokens=497, prompt_tokens=19, total_tokens=516, completion_time=1.284815074, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=324), prompt_time=0.000574971, prompt_tokens_details=None, queue_time=0.046935688, total_time=1.285390045)